In [1]:
import xarray as xr
import numpy as np

In [2]:
%matplotlib qt5

QStandardPaths: error creating runtime directory '/run/user/2784' (Permission denied)


In [ ]:
inputpath_raw = '/data/cburgard/PREPARE_FORCING/PREPARE_PISCES/raw/'
inputpath_interim = '/data/cburgard/PREPARE_FORCING/PREPARE_PISCES/interim/'

In [ ]:
file_orig = xr.open_dataset(inputpath_raw + 'etopo2bedmap.nc')

In [ ]:
file_orig

In [ ]:
lat = ((1./30.)*(file_orig.jpjo)+1./30.-90    )*-1

In [ ]:
lon = (-180.)+1./30.*(file_orig.jpio)

In [ ]:
file_new = file_orig.copy()
file_new = file_new.rename({'jpjo': 'lat', 'jpio': 'lon'}).assign_coords({'lat':lat.values, 'lon':lon.values})

In [ ]:
file_new['depth'].plot()

In [ ]:
file_new.lon

In [ ]:
file_gebco = xr.open_dataset(inputpath_raw + 'GEBCO_2024_sub_ice_topo.nc')

In [ ]:
diff_lon = 180-179.997917

In [ ]:
diff_lon

In [ ]:
file_new.lon

In [ ]:
240/30

In [ ]:
file_gebco_resampled = file_gebco.sel(lon=file_gebco.lon[::8],lat=file_gebco.lat[::8])
file_gebco_resampled = file_gebco_resampled.reindex(lat=list(reversed(file_gebco_resampled.lat)))

In [ ]:
file_new.lat.values - file_gebco_resampled.lat.values

In [ ]:
file_new2 = file_orig.copy()

In [ ]:
file_new2['depth'] = xr.DataArray(data=file_gebco_resampled['elevation'].values, dims=file_orig.dims)

In [ ]:
file_new2['depth'] = file_new2['depth'].where(file_new2['depth'] < 0,0)

In [ ]:
file_new2.to_netcdf(inputpath_interim + 'gebco_2024_inetopo2bedmap_format.nc')

In [ ]:
file_new2_cut_Ant = file_new2['depth'].where((file_new2['depth'] > -2000) & (file_new2.jpjo > 4500))

In [ ]:
file_new_merged = file_new2.copy()
file_new_merged['depth'] = file_new2['depth'].where((file_new2['depth'] > -2000) & (file_new2.jpjo > 4500), file_orig['depth'])

In [ ]:
file_new_merged.to_netcdf(inputpath_interim + 'etopo2bedmap_mergedwith_gebco2024_for_Ant.nc')

In [ ]:
file_new_merged['depth'].plot()

In [ ]:
file_gebco_new = file_gebco.interp2({'lon':file_new.lon,'lat':file_new.lat})

In [ ]:
### ADD A HALO TO THE mesh_mask

In [57]:
inputpath_thredds = '/thredds/tgcc/work/burgardc/'
outputpath = '/data/cburgard/PREPARE_FORCING/PREPARE_PISCES/interim/'

In [4]:
mesh_mask = xr.open_dataset(inputpath_thredds + 'eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights_mesh_mask_withz2.nc')

In [7]:
if 'x' in mesh_mask.tmask.dims:
    print('yes')

yes


In [10]:
tmask = mesh_mask['tmask']

In [22]:
tmask.x.max().values

array(360)

In [15]:
tmask['x'] = tmask['x'] + 1
tmask['y'] = tmask['y'] + 1

In [33]:
tmask_left = xr.DataArray(data=tmask.sel(x=[1]).values, dims=tmask.dims).assign_coords({'x': [0], 'y': tmask.y, 'z': tmask.z, 'time_counter': tmask.time_counter})
tmask_right = xr.DataArray(data=tmask.sel(x=[360]).values, dims=tmask.dims).assign_coords({'x': [361], 'y': tmask.y, 'z': tmask.z, 'time_counter': tmask.time_counter})
tmask_halox = xr.concat([tmask_left,tmask,tmask_right], dim='x')

In [34]:
tmask_bottom = xr.DataArray(data=tmask_halox.sel(y=[1]).values, dims=tmask_halox.dims).assign_coords({'x': tmask_halox.x, 'y': [0], 'z': tmask_halox.z, 'time_counter': tmask_halox.time_counter})
tmask_halox_and_y = xr.concat([tmask_bottom,tmask_halox], dim='y')

In [54]:
mesh_mask_halo = mesh_mask.copy()
mesh_mask_halo['x'] = mesh_mask_halo['x'] + 1
mesh_mask_halo['y'] = mesh_mask_halo['y'] + 1

halo_list = []
for vvar in list(mesh_mask.keys()):
    print(vvar)
    if 'x' in mesh_mask[vvar].dims:
        vv_left = xr.DataArray(data=mesh_mask_halo[vvar].sel(x=[1]).values, dims=mesh_mask_halo[vvar].dims).assign_coords({'x': [0]}) 
                                                                                                           #                'y': mesh_mask_halo.y, 
                                                                                                           #                'z': mesh_mask_halo.z, 
                                                                                                           #                'time_counter': mesh_mask_halo.time_counter})
        vv_right = xr.DataArray(data=mesh_mask_halo[vvar].sel(x=[360]).values, dims=mesh_mask_halo[vvar].dims).assign_coords({'x': [361]}) 
                                                                                                              #                'y': mesh_mask_halo.y, 
                                                                                                              #               'z': mesh_mask_halo.z, 
                                                                                                              #                'time_counter': mesh_mask_halo.time_counter})
        vv_halox = xr.concat([vv_left,mesh_mask_halo[vvar],vv_right], dim='x')

        vv_bottom = xr.DataArray(data=vv_halox.sel(y=[1]).values, dims=vv_halox.dims).assign_coords({'y': [0]}) #, 'x': mesh_mask_halo.x, 'z': mesh_mask_halo.z, 'time_counter': mesh_mask_halo.time_counter})
        vv_halox_and_y = xr.concat([vv_bottom,vv_halox], dim='y')
        halo_list.append(vv_halox_and_y.rename(vvar)) 

    else:
        halo_list.append(mesh_mask[vvar])

mesh_mask_halo_new = xr.merge(halo_list)

bathy_metry
e1f
e1t
e1u
e1v
e2f
e2t
e2u
e2v
e3t_0
e3u_0
e3v_0
e3w_0
ff_f
ff_t
fmask
gdept_0
gdept_1d
gdepw_0
gdepw_1d
glamf
glamt
glamu
glamv
gphif
gphit
gphiu
gphiv
hw
isfdraft
jperio
jpiglo
jpjglo
jpkglo
ln_isfcav
ln_sco
ln_zco
ln_zps
mbathy
mhw
misf
nav_lat
nav_lev
nav_lon
strait_shlat
tmask
tmaskutil
umask
umaskutil
vmask
vmaskutil


In [59]:
mesh_mask_halo_new.to_netcdf(outputpath + 'eORCA1.4.3_OpenSeas_OpenAllCav_ModStraights_mesh_mask_withz2_andhalo.nc', unlimited_dims=['time_counter'])

In [61]:
mesh_mask_halo_new['tmask'].isel(z=0).plot()

In [44]:
xr.DataArray(data=mesh_mask_halo[vvar].sel(x=[1]).values, dims=mesh_mask_halo[vvar].dims).x

<xarray.DataArray 'x' (x: 1)>
array([0])
Dimensions without coordinates: x